# ComfyUI + FLUX.1 Kontext on Google Colab

Notebook setup ComfyUI với FLUX.1 Kontext stack chạy trên Colab Pro (T4 16GB GPU), expose qua cloudflared HTTPS tunnel để BE local (FastAPI ở `e:\Python\DATN-_HaUI\backend`) gọi vào.

## Cách dùng

1. **Runtime → Change runtime type → GPU** (T4 hoặc V100).
2. **Lần đầu**: Run all. Cell 2 sẽ tải ~10GB models vào Google Drive của mày (~1-3 phút từ HF, tốc độ Colab→HF khá nhanh). Subsequent sessions cell 2 sẽ skip.
3. **Cell cuối cùng** in ra `COMFY_BASE_URL=https://....trycloudflare.com` — paste vào `backend/.env`, restart BE bằng `run_dev.bat`.
4. **Đừng đóng tab Colab** trong khi dùng — cell heartbeat chạy mãi để giữ session sống.
5. Stop bằng cách click nút stop ở cell heartbeat — tự cleanup tunnel + ComfyUI.

## Stack model FLUX.1 Kontext Q3_K_S

| File | Size | Vai trò |
|---|---|---|
| `flux1-kontext-dev-Q3_K_S.gguf` | 5.2GB | UNet (diffusion model, GGUF Q3 quant) |
| `t5xxl_fp8_e4m3fn.safetensors` | 4.9GB | T5xxl text encoder (fp8) |
| `clip_l.safetensors` | 235MB | CLIP-L text encoder |
| `ae.safetensors` | 335MB | FLUX VAE |

Tổng disk: ~10.7GB. Phù hợp T4 16GB (Colab) và RTX 3060 12GB (rent sau).

## Sự khác biệt với z-image-turbo workflow ở local

FLUX dùng T5 + CLIP-L (KHÔNG dùng qwen_3_4b). VAE thì cùng `ae.safetensors`. UNet khác hẳn (FLUX architecture vs z-image DiT). Sẽ cần workflow JSON riêng cho BE Phase 7 — tao sẽ viết khi mày confirm Colab chạy được.

## Bước 1: Mount Google Drive

Drive dùng làm persistent storage. Mỗi session Colab `/content/` bị wipe, nhưng Drive thì giữ nguyên — model tải 1 lần, dùng mãi.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/comfyui_models')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## Bước 2: Tải model FLUX Kontext stack vào Drive (chỉ chạy lần đầu)

Subsequent runs sẽ skip nếu file đã tồn tại trên Drive với size đúng. Speed download trong Colab ~100-300MB/s nên tổng ~1-3 phút lần đầu.

In [ ]:
import os
from pathlib import Path

MODELS = [
    (
        'diffusion_models/flux1-kontext-dev-Q3_K_S.gguf',
        'https://huggingface.co/QuantStack/FLUX.1-Kontext-dev-GGUF/resolve/main/flux1-kontext-dev-Q3_K_S.gguf',
        5200,
    ),
    (
        'clip/t5xxl_fp8_e4m3fn.safetensors',
        'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
        4900,
    ),
    (
        'clip/clip_l.safetensors',
        'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
        230,
    ),
    (
        'vae/ae.safetensors',
        'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors',
        320,
    ),
]

for rel_path, url, expected_mb in MODELS:
    target = DRIVE_ROOT / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        actual_mb = target.stat().st_size / (1024 * 1024)
        if actual_mb > expected_mb * 0.95:
            print(f'[SKIP] {rel_path} ({actual_mb:.0f}MB) already on Drive')
            continue
        print(f'[WARN] {rel_path} size {actual_mb:.0f}MB != expected ~{expected_mb}MB, re-downloading')
    print(f'[DOWNLOAD] {rel_path} (~{expected_mb}MB)')
    os.system(f'wget -q --show-progress -O "{target}" "{url}"')

print('\nAll models on Drive.')

## Bước 3: Cài ComfyUI + custom node ComfyUI-GGUF

ComfyUI clone vào `/content/` (mất khi session reset, OK). GGUF custom node để load `.gguf` UNet.

In [ ]:
import os
import subprocess

if not os.path.exists('/content/ComfyUI'):
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI', '/content/ComfyUI'],
        check=True,
    )
    print('[OK] ComfyUI cloned')
else:
    print('[SKIP] ComfyUI already cloned')

print('Installing ComfyUI requirements...')
subprocess.run(
    ['pip', 'install', '-q', '-r', '/content/ComfyUI/requirements.txt'],
    check=True,
)

GGUF_DIR = '/content/ComfyUI/custom_nodes/ComfyUI-GGUF'
if not os.path.exists(GGUF_DIR):
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/city96/ComfyUI-GGUF', GGUF_DIR],
        check=True,
    )
    print('[OK] ComfyUI-GGUF cloned')
else:
    print('[SKIP] ComfyUI-GGUF already cloned')

subprocess.run(
    ['pip', 'install', '-q', '-r', f'{GGUF_DIR}/requirements.txt'],
    check=True,
)
print('[OK] All deps installed')

## Bước 4: Symlink models từ Drive vào ComfyUI

ComfyUI sẽ đọc model trực tiếp từ Drive qua symlink. Lần load đầu tiên trong session sẽ chậm hơn ~50-90s (Drive read ~50-100MB/s vs SSD local 3-5GB/s), nhưng sau đó model cache trong RAM/VRAM, gen sau nhanh bình thường.

In [ ]:
from pathlib import Path

COMFY_MODELS = Path('/content/ComfyUI/models')

for subdir in ['diffusion_models', 'clip', 'vae']:
    drive_dir = DRIVE_ROOT / subdir
    comfy_dir = COMFY_MODELS / subdir
    comfy_dir.mkdir(parents=True, exist_ok=True)
    if not drive_dir.exists():
        print(f'[WARN] {drive_dir} not found, skip')
        continue
    for f in drive_dir.iterdir():
        if not f.is_file():
            continue
        link = comfy_dir / f.name
        if link.is_symlink() or link.exists():
            link.unlink()
        link.symlink_to(f)
        print(f'[LINK] {subdir}/{f.name}')

## Bước 5: Cài cloudflared

Cloudflared quick tunnel cho HTTPS URL public, hỗ trợ cả HTTP và WebSocket. URL đổi mỗi session — paste lại vào BE `.env` mỗi lần.

In [ ]:
import os
import subprocess

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(
        [
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', '/usr/local/bin/cloudflared',
        ],
        check=True,
    )
    os.chmod('/usr/local/bin/cloudflared', 0o755)
    print('[OK] cloudflared installed')
else:
    print('[SKIP] cloudflared already installed')

## Bước 6: Start ComfyUI in background

Wait đến khi ComfyUI respond `/system_stats` thì coi như ready (~30-60s cho lần đầu, ~20s cho subsequent).

In [ ]:
import subprocess
import time
import urllib.request

COMFY_LOG = '/content/comfyui.log'
_log_fp = open(COMFY_LOG, 'w')

comfy_proc = subprocess.Popen(
    ['python', '/content/ComfyUI/main.py', '--listen', '0.0.0.0', '--port', '8188'],
    stdout=_log_fp, stderr=subprocess.STDOUT,
    cwd='/content/ComfyUI',
)
print(f'ComfyUI starting (pid {comfy_proc.pid})... log at {COMFY_LOG}')

ready = False
for i in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        ready = True
        break
    except Exception:
        time.sleep(1)

if not ready:
    print('[FAIL] ComfyUI did not respond in 120s. Last 30 log lines:')
    os.system(f'tail -30 {COMFY_LOG}')
else:
    print(f'[OK] ComfyUI ready after {i+1}s')

## Bước 7: Start cloudflared tunnel, lấy URL

Sau cell này, **copy URL `https://...trycloudflare.com`** in ra dưới và paste vào `backend/.env` ở máy local:

```
COMFY_BASE_URL=https://<id>.trycloudflare.com
```

Restart BE: `cd backend && run_dev.bat`.

In [ ]:
import subprocess
import re
import time

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
print('Cloudflared tunnel starting...')

tunnel_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
    if m:
        tunnel_url = m.group(1)
        break

if tunnel_url:
    print()
    print('=' * 72)
    print(f'COMFY_BASE_URL={tunnel_url}')
    print('=' * 72)
    print()
    print('Paste line trên (1 dòng) vào backend/.env, ghi đè COMFY_BASE_URL hiện có.')
    print('Sau đó restart BE: cd backend && run_dev.bat')
    print()
    print('URL chỉ tồn tại đến khi tunnel stop hoặc session Colab die.')
else:
    print('[FAIL] Không lấy được URL trong 60s. Xem output cloudflared phía trên.')

## Bước 8: Heartbeat — giữ session sống

Cell dưới chạy vô hạn, in heartbeat mỗi 10 phút. Đừng tắt cell này khi đang dùng ComfyUI từ BE local.

**Stop**: click nút stop ở cell — sẽ tự cleanup tunnel + ComfyUI.

In [ ]:
import time
from datetime import datetime, timezone

start = datetime.now(timezone.utc)
print(f'Heartbeat started at {start.strftime("%Y-%m-%d %H:%M:%S UTC")}')
print(f'Tunnel: {tunnel_url}')
print()

try:
    while True:
        time.sleep(600)
        elapsed_h = (datetime.now(timezone.utc) - start).total_seconds() / 3600
        now = datetime.now(timezone.utc).strftime('%H:%M:%S')
        print(f'[{now} UTC] Alive — {elapsed_h:.1f}h since start')
except KeyboardInterrupt:
    print('\nStopping...')
    tunnel_proc.terminate()
    comfy_proc.terminate()
    print('Tunnel + ComfyUI stopped. URL no longer valid.')